# **Step 1 : Dependencies Installation**

In [2]:
%pip install ultralytics torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
Note: you may need to restart the kernel to use updated packages.


# **Step 2 : Model_Evalution** 

In [7]:
from ultralytics import YOLO

# Load model (explicit task)
model = YOLO(
    "/kaggle/input/model/pytorch/default/1/best.pt",
    task="detect"
)

# Run validation
metrics = model.val(
    data="/kaggle/input/kaggle-yaml/kaggle_new_data.yaml",
    imgsz=640,
    batch=16,
    device=0,          # single GPU only
    split="val",
    save_json=True,
    plots=True,
    verbose=True,
    visualize=True     # optional (slow)
)

# Print metrics
print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("mAP75:", metrics.box.map75)
print("Per-class mAP:", metrics.box.maps)

# Confusion matrix as DataFrame
df_cm = metrics.confusion_matrix.to_df()
print(df_cm)

Ultralytics 8.3.250 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,848,445 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.2 ms, read: 57.5±24.6 MB/s, size: 464.8 KB)
val: Scanning /kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset/val/labels... 4196 images, 9 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4196/4196 192.9it/s 21.7s<0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset/val is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 263/263 1.0it/s 4:21<0.7s
                   all       4196      49838       0.73      0.431      0.472      0.313
                animal        219        753      0.663      0.216      0.281      0.129
          autorickshaw       1436       3205      0.817      0.649      0.713       0.51
    

# **Step 3 : CSV File Creation**

In [13]:
import os

# Convert metrics to CSV string
val_csv = metrics.to_csv()
print(val_csv)

# Define output path
dir_path = "/kaggle/working/runs/detect"
csv_filename = "validation_results.csv"

# Ensure directory exists
os.makedirs(dir_path, exist_ok=True)

# Full file path
full_path = os.path.join(dir_path, csv_filename)

# Write CSV
with open(full_path, "w") as f:
    f.write(val_csv)

print("Saved at:", full_path)


Class,Images,Instances,Box-P,Box-R,Box-F1,mAP50,mAP50-95
animal,219,753,0.66311,0.21647,0.32639,0.2814,0.12884
autorickshaw,1436,3205,0.81659,0.64875,0.72306,0.71284,0.50995
bicycle,264,301,0.65336,0.44186,0.52719,0.47429,0.27801
bus,1061,1794,0.80519,0.62486,0.70365,0.68818,0.52765
car,2582,8793,0.81629,0.59252,0.68663,0.65934,0.46495
caravan,18,18,0.49762,0.72222,0.58924,0.56935,0.54212
motorcycle,2778,10059,0.79344,0.6094,0.68935,0.66451,0.40094
person,2105,8863,0.7752,0.41149,0.53761,0.49243,0.26939
rider,2454,9444,0.77198,0.4897,0.59927,0.56393,0.32994
traffic light,157,370,0.59843,0.24324,0.34589,0.31447,0.16052
traffic sign,774,1404,0.65191,0.32621,0.43483,0.36127,0.19635
train,4,4,1.0,0.0,0.0,0.0013,0.00026
truck,1564,2758,0.7703,0.59214,0.66957,0.6644,0.4802
vehicle fallback,1116,2072,0.6117,0.12017,0.20088,0.16,0.08846

Saved at: /kaggle/working/runs/detect/validation_results.csv


# **Step 4 : Zip File Creation** 

In [14]:
import shutil
import os

folder_path = "/kaggle/working/runs/detect"
zip_path = "/kaggle/working/evaluation_results"

# Remove existing zip if it exists
if os.path.exists(zip_path + ".zip"):
    os.remove(zip_path + ".zip")

# Create zip archive
shutil.make_archive(zip_path, 'zip', folder_path)

print("ZIP file created at:", zip_path + ".zip")

ZIP file created at: /kaggle/working/evaluation_results.zip


# **Step 5 : Model BenchMarking**

In [29]:
from ultralytics.utils.benchmarks import benchmark

benchmark(
    model="/kaggle/input/15best-pt/pytorch/default/1/best.pt",
    data="/kaggle/input/kaggle-yaml/kaggle_new_data.yaml",
    imgsz=640,
    half=False,
    device=0,
)


Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6636.8/8062.4 GB disk)

Benchmarks complete for /kaggle/input/15best-pt/pytorch/default/1/best.pt on /kaggle/input/kaggle-yaml/kaggle_new_data.yaml at imgsz=640 (108.05s)
Benchmarks legend:  - ✅ Success  - ❎ Export passed but validation failed  - ❌️ Export failed
+----------------------------------------------------------------------------------------------------------+
|      Format                  Status❔   Size (MB)   metrics/mAP50-95(B)   Inference time (ms/im)   FPS   |
+==========================================================================================================+
| 1    PyTorch                 ✅         49.6        0.3127                13.82                    72.34 |
| 2    TorchScript             ❌         0.0         -                     -                        -     |
| 3    ONNX                    ❌         0.0         -                     -                        -     |
| 4    OpenVINO                ❌         0.0   

,Format,Status❔,Size (MB),metrics/mAP50-95(B),Inference time (ms/im),FPS
"""1""","""PyTorch""","""✅""","""49.6""","""0.3127""","""13.82""","""72.34"""
"""2""","""TorchScript""","""❌""","""0.0""","""-""","""-""","""-"""
"""3""","""ONNX""","""❌""","""0.0""","""-""","""-""","""-"""
"""4""","""OpenVINO""","""❌""","""0.0""","""-""","""-""","""-"""
"""5""","""TensorRT""","""❌""","""0.0""","""-""","""-""","""-"""
"""6""","""CoreML""","""❌""","""0.0""","""-""","""-""","""-"""
"""7""","""TensorFlow SavedModel""","""❌""","""0.0""","""-""","""-""","""-"""
"""8""","""TensorFlow GraphDef""","""❌""","""0.0""","""-""","""-""","""-"""
"""9""","""TensorFlow Lite""","""❌""","""0.0""","""-""","""-""","""-"""
"""10""","""TensorFlow Edge TPU""","""❌""","""0.0""","""-""","""-""","""-"""
